# Estudo de Caso 12.2 — Separação Cromatográfica Glicose/Frutose: Modelo Cinético de Primeira Ordem

| Campo | Detalhe |
|:------|:--------|
| **Capítulo Associado** | Capítulo 12 — Adsorção Interfacial: Colunas de Bolhas e Cromatografia |
| **Nível** | Ambos (Graduação e Pós-Graduação) |
| **Tempo estimado** | 75 minutos |
| **Autor** | Jader Lugon Junior |
| **Método numérico** | Euler Melhorado (Runge-Kutta de 2ª ordem) |
| **Bibliografia** | Oliveira et al. (2017); Lugon Jr. et al. (2026), Cap. 12 |

---

## 📋 Resumo

Modelamos a separação cromatográfica de glicose e frutose em coluna de leito fixo. O modelo utiliza cinética de primeira ordem para a transferência de massa entre a fase líquida (móvel) e a fase sólida (estacionária). A separação é possível porque a frutose tem maior afinidade pela fase estacionária ($K_{fru} \approx 1{,}09$) que a glicose ($K_{glic} \approx 0{,}54$), resultando em seletividade $\alpha \approx 2{,}0$. As equações acopladas são integradas pelo método de Euler Melhorado, produzindo os cromatogramas (perfis de eluição) que caracterizam a separação.

---

## 🎯 Objetivos de Aprendizagem

Ao final deste estudo de caso, o leitor será capaz de:

1. **Implementar** o modelo cinético de primeira ordem para adsorção em leito fixo
2. **Resolver** o sistema de EDOs acopladas pelo método de Euler Melhorado
3. **Calcular** a afinidade de equilíbrio $K_j = k_{1j}/k_{2j}$ e a seletividade $\alpha$
4. **Interpretar** cromatogramas: tempo de pico (afinidade) e largura de pico (cinética)
5. **Analisar** o efeito das constantes cinéticas $k_1$ e $k_2$ sobre a separação
6. **Avaliar** a qualidade da separação em função da seletividade
7. **Discutir** o princípio de operação do Leito Móvel Simulado (LMS)

---
## ⚙️ Seção 0: Configuração Inicial

In [ ]:
# ============================================================
# CONFIGURAÇÃO INICIAL — Importação de bibliotecas
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display, Markdown
import time

# Configuração de estilo dos gráficos (padrão do guia)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print('✅ Ambiente configurado com sucesso')
print(f'📦 NumPy {np.__version__} | Matplotlib {plt.matplotlib.__version__}')

---
## 🧪 Seção 1: Parâmetros da Separação Cromatográfica

### Contexto Físico

Na **cromatografia em coluna de leito fixo**, a fase líquida (móvel) escoa através de um leito de partículas sólidas (fase estacionária). Os componentes da mistura são separados porque têm diferentes afinidades pela fase estacionária:

- **Baixa afinidade** (glicose): permanece na fase móvel, elui mais cedo
- **Alta afinidade** (frutose): passa mais tempo adsorvida, elui mais tarde

A **separação glicose/frutose** é industrialmente relevante para produção de xarope de milho de alta frutose (HFCS), utilizado como adoçante na indústria alimentícia.

> 💡 **Observação:** Em aplicações industriais, o processo contínuo é realizado em equipamentos de **Leito Móvel Simulado (LMS)**, que simulam o fluxo contracorrente das fases sólida e líquida por meio do chaveamento periódico das portas de entrada e saída em cada coluna.

### Modelo Cinético de Primeira Ordem

$$\frac{dC_j}{dt} = -k_{1j} C_j + k_{2j} q_j$$

$$\frac{dq_j}{dt} = +k_{1j} C_j - k_{2j} q_j$$

onde:
- $C_j$ = concentração na fase líquida [mg/mL]
- $q_j$ = concentração na fase sólida [mg/mL]
- $k_{1j}$ = constante cinética de adsorção [min⁻¹]
- $k_{2j}$ = constante cinética de dessorção [min⁻¹]
- $K_j = k_{1j}/k_{2j}$ = afinidade de equilíbrio

### Tabela de Parâmetros

| Espécie | $k_1$ [min⁻¹] | $k_2$ [min⁻¹] | $K = k_1/k_2$ | Eluição |
|:--------|:---:|:---:|:---:|:---|
| Glicose | 0,54 | 1,00 | 0,54 | Primeiro |
| Frutose | 1,09 | 1,00 | 1,09 | Depois |
| **Seletividade** | – | – | $\alpha = 1{,}09/0{,}54 \approx 2{,}0$ | – |

> 📖 **Referência:** Oliveira et al. (2017); Lugon Jr. et al. (2026), Cap. 12, Seção 12.4.

In [ ]:
# ============================================================
# PARÂMETROS DA SEPARAÇÃO CROMATOGRÁFICA
# ============================================================

# Constantes cinéticas (primeira ordem)
k1_glic = 0.54       # Adsorção da glicose [min⁻¹]
k2_glic = 1.00       # Dessorção da glicose [min⁻¹]
k1_fru = 1.09        # Adsorção da frutose [min⁻¹]
k2_fru = 1.00        # Dessorção da frutose [min⁻¹]

# Afinidades de equilíbrio
K_glic = k1_glic / k2_glic   # ≈ 0,54
K_fru = k1_fru / k2_fru      # ≈ 1,09
alpha_sel = K_fru / K_glic   # Seletividade ≈ 2,0

# Condições iniciais (injeção em banda)
C_glic_0 = 1.0       # Concentração inicial de glicose [mg/mL]
C_fru_0 = 1.0        # Concentração inicial de frutose [mg/mL]
q_glic_0 = 0.0       # Fase sólida inicialmente limpa
q_fru_0 = 0.0

# Parâmetros de simulação
t_final = 10.0       # Tempo final [min]
dt = 0.001           # Passo de tempo [min]
N_steps = int(t_final / dt)

print('📏 Parâmetros da Separação Cromatográfica:')
print(f'  • Glicose: k₁ = {k1_glic}, k₂ = {k2_glic}, K = {K_glic:.2f}')
print(f'  • Frutose: k₁ = {k1_fru}, k₂ = {k2_fru}, K = {K_fru:.2f}')
print(f'  • Seletividade: α = K_fru/K_glic = {alpha_sel:.2f}')
print(f'  • Condições iniciais: C_glic = C_fru = 1,0 mg/mL')
print(f'  • Simulação: t_f = {t_final} min, Δt = {dt} min, N = {N_steps} passos')

---
## 🔬 Seção 2: Verificação — Conservação de Massa

Antes da simulação, verificamos que o modelo conserva massa.

Para cada espécie $j$, a massa total (líquido + sólido) deve ser constante:

$$C_j(t) + q_j(t) = C_j(0) + q_j(0) = \text{constante}$$

Isso decorre diretamente da estrutura das equações acopladas:

$$\frac{dC_j}{dt} + \frac{dq_j}{dt} = 0$$

> 📖 **Referência:** Oliveira et al. (2017); Lugon Jr. et al. (2026), Cap. 12.

In [ ]:
# ============================================================
# VERIFICAÇÃO — CONSERVAÇÃO DE MASSA
# ============================================================

print('🔬 VERIFICAÇÃO — CONSERVAÇÃO DE MASSA')
print('=' * 50)

# A estrutura das equações garante dC/dt + dq/dt = 0
# Verificar algebricamente:
# dC/dt = -k1*C + k2*q
# dq/dt = +k1*C - k2*q
# Soma: 0 ✓

print(f'  • dC/dt + dq/dt = (-k1·C + k2·q) + (k1·C - k2·q) = 0 ✓')
print(f'  • Massa total de glicose: C_glic + q_glic = {C_glic_0 + q_glic_0} mg/mL (constante)')
print(f'  • Massa total de frutose: C_fru + q_fru = {C_fru_0 + q_fru_0} mg/mL (constante)')
print(f'\n✅ Modelo conserva massa por construção')

---
## 📐 Seção 3: Formulação Matemática

### 3.1 Sistema de EDOs Acopladas

Para cada espécie $j \in \{glic, fru\}$:

$$\frac{dC_j}{dt} = -k_{1j} C_j + k_{2j} q_j$$

$$\frac{dq_j}{dt} = +k_{1j} C_j - k_{2j} q_j$$

### 3.2 Solução Analítica de Equilíbrio

Em equilíbrio ($dC/dt = dq/dt = 0$):

$$q_j^{eq} = K_j \cdot C_j^{eq}, \quad K_j = \frac{k_{1j}}{k_{2j}}$$

### 3.3 Interpretação Física dos Parâmetros

| Parâmetro | Efeito Físico | Observável |
|:----------|:--------------|:-----------|
| $k_1$ (adsorção) | Velocidade de captura pela fase sólida | Largura do pico |
| $k_2$ (dessorção) | Velocidade de liberação da fase sólida | Largura do pico |
| $K = k_1/k_2$ | Afinidade de equilíbrio | Tempo de pico (retenção) |
| $\alpha = K_{fru}/K_{glic}$ | Seletividade da separação | Distância entre picos |

> 📖 **Referência:** Oliveira et al. (2017); Lugon Jr. et al. (2026), Cap. 12, Seção 12.4.

---
## ⚡ Seção 4: Critério Numérico

> 💡 **Observação:** Neste sistema de EDOs lineares acopladas, os critérios de adequação são:

1. **Passo de tempo:** $\Delta t$ deve ser pequeno comparado ao tempo característico $\tau = 1/(k_1 + k_2)$. Para $k_1 + k_2 \approx 2$ min⁻¹, $\tau \approx 0{,}5$ min. Usamos $\Delta t = 0{,}001$ min (500× menor).

2. **Método de Euler Melhorado (RK2):** Erro de truncamento $\mathcal{O}(\Delta t^3)$, adequado para a suavidade dos perfis de eluição.

3. **Conservação de massa:** Verificada a cada passo: $C_j + q_j = $ constante.

4. **Estabilidade:** O sistema é linear com autovalores negativos, portanto incondicionalmente estável para métodos explícitos com $\Delta t$ suficientemente pequeno.

---
## 🌪️ Seção 5: Condições de Entrada (Forçamento)

Neste problema, o forçamento é a **injeção em banda** da mistura:

$$C_{glic}(0) = 1{,}0 \text{ mg/mL}, \quad C_{fru}(0) = 1{,}0 \text{ mg/mL}$$

$$q_{glic}(0) = 0, \quad q_{fru}(0) = 0 \quad \text{(leito limpo)}$$

> 💡 **Observação:** Na prática cromatográfica, a injeção em banda é seguida por uma fase de eluição com solvente puro. Neste modelo simplificado, a dinâmica de adsorção/dessorção já produz a separação das bandas.

---
## 🧮 Seção 6: Implementação Numérica

In [ ]:
# ============================================================
# FUNÇÃO: LADO DIREITO DO SISTEMA DE EDOs
# ============================================================

def sistema_cromatografico(y, k1, k2):
    """
    Lado direito do sistema de EDOs para uma espécie.

    Parameters
    ----------
    y : ndarray
        Vetor [C_j, q_j] — concentrações nas fases líquida e sólida.
    k1 : float
        Constante cinética de adsorção [min⁻¹].
    k2 : float
        Constante cinética de dessorção [min⁻¹].

    Returns
    -------
    ndarray
        Derivadas [dC/dt, dq/dt].
    """
    C, q = y
    dCdt = -k1 * C + k2 * q
    dqdt = k1 * C - k2 * q
    return np.array([dCdt, dqdt])


def solver_euler_melhorado(y0, k1, k2, t_final, dt):
    """
    Resolve o sistema de EDOs pelo método de Euler Melhorado (RK2).

    Parameters
    ----------
    y0 : ndarray
        Condição inicial [C_j(0), q_j(0)].
    k1 : float
        Constante cinética de adsorção [min⁻¹].
    k2 : float
        Constante cinética de dessorção [min⁻¹].
    t_final : float
        Tempo final [min].
    dt : float
        Passo de tempo [min].

    Returns
    -------
    tuple
        (t_arr, C_arr, q_arr) — vetores de tempo e concentrações.
    """
    N_steps = int(t_final / dt)
    t_arr = np.zeros(N_steps + 1)
    y_arr = np.zeros((N_steps + 1, 2))

    y_arr[0] = y0

    for n in range(N_steps):
        t_arr[n+1] = (n + 1) * dt

        # Estágio 1
        k1_stage = sistema_cromatografico(y_arr[n], k1, k2)

        # Estágio 2 (ponto médio)
        y_mid = y_arr[n] + 0.5 * dt * k1_stage
        k2_stage = sistema_cromatografico(y_mid, k1, k2)

        # Atualização
        y_arr[n+1] = y_arr[n] + dt * k2_stage

    return t_arr, y_arr[:, 0], y_arr[:, 1]


print('✅ Solver de Euler Melhorado implementado')

In [ ]:
# ============================================================
# SIMULAÇÃO DA SEPARAÇÃO CROMATOGRÁFICA
# ============================================================

t_inicio = time.time()

# Glicose
y0_glic = np.array([C_glic_0, q_glic_0])
t_arr, C_glic, q_glic = solver_euler_melhorado(
    y0_glic, k1_glic, k2_glic, t_final, dt)

# Frutose
y0_fru = np.array([C_fru_0, q_fru_0])
_, C_fru, q_fru = solver_euler_melhorado(
    y0_fru, k1_fru, k2_fru, t_final, dt)

t_cpu_sim = time.time() - t_inicio

# Verificação de conservação de massa
massa_glic = C_glic + q_glic
massa_fru = C_fru + q_fru
erro_massa_glic = np.max(np.abs(massa_glic - massa_glic[0]))
erro_massa_fru = np.max(np.abs(massa_fru - massa_fru[0]))

print('=' * 60)
print('  SIMULAÇÃO CONCLUÍDA')
print('=' * 60)
print(f'\n📊 Glicose:')
print(f'  • C(0) = {C_glic[0]:.4f} mg/mL')
print(f'  • C(t_f) = {C_glic[-1]:.6f} mg/mL')
print(f'  • q(t_f) = {q_glic[-1]:.6f} mg/mL')
print(f'\n📊 Frutose:')
print(f'  • C(0) = {C_fru[0]:.4f} mg/mL')
print(f'  • C(t_f) = {C_fru[-1]:.6f} mg/mL')
print(f'  • q(t_f) = {q_fru[-1]:.6f} mg/mL')
print(f'\n✅ Conservação de massa:')
print(f'  • Erro máximo (glicose): {erro_massa_glic:.2e}')
print(f'  • Erro máximo (frutose): {erro_massa_fru:.2e}')
print(f'\n⏱️ Tempo de CPU: {t_cpu_sim:.4f} s')

---
## 📊 Seção 7: Visualização

In [ ]:
# ============================================================
# GRÁFICO 1: CROMATOGRAMAS (PERFIS DE ELUIÇÃO)
# ============================================================

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(t_arr, C_glic, 'b-', linewidth=2.5, label='Glicose ($K$ = 0,54)')
ax.plot(t_arr, C_fru, 'r-', linewidth=2.5, label='Frutose ($K$ = 1,09)')

# Marcar tempos de pico
t_pico_glic = t_arr[np.argmax(C_glic)]
t_pico_fru = t_arr[np.argmax(C_fru)]

ax.axvline(t_pico_glic, color='blue', linestyle=':', linewidth=1.5, alpha=0.5,
           label=f'Pico glicose: t = {t_pico_glic:.2f} min')
ax.axvline(t_pico_fru, color='red', linestyle=':', linewidth=1.5, alpha=0.5,
           label=f'Pico frutose: t = {t_pico_fru:.2f} min')

ax.set_xlabel('Tempo de retenção [min]', fontsize=12)
ax.set_ylabel('Concentração $C$ [mg/mL]', fontsize=12)
ax.set_title('Cromatograma — Separação Glicose/Frutose',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n📊 Interpretação do cromatograma:')
print(f'  • Tempo de pico (glicose): {t_pico_glic:.2f} min — menor afinidade, elui primeiro')
print(f'  • Tempo de pico (frutose): {t_pico_fru:.2f} min — maior afinidade, elui depois')
print(f'  • Separação temporal: {t_pico_fru - t_pico_glic:.2f} min')

> 💡 **Observação:**
> - A **glicose** (menor afinidade, $K = 0{,}54$) elui primeiro: passa mais tempo na fase móvel e avança mais rapidamente pela coluna.
> - A **frutose** (maior afinidade, $K = 1{,}09$) elui depois: passa mais tempo adsorvida na fase estacionária.
> - O **pico da frutose** é mais baixo e largo porque a espécie permanece mais tempo na coluna, sujeita ao espalhamento cinético.
> - Com $\alpha \approx 2{,}0$, os picos ficam suficientemente afastados para permitir coleta fracionada.
> - **Leitura do gráfico:** o tempo de pico codifica a afinidade $K_j$; a largura do pico codifica a cinética ($k_1$, $k_2$).

In [ ]:
# ============================================================
# GRÁFICO 2: PERFIS DE FASE SÓLIDA
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Fase líquida vs. sólida (glicose)
ax1.plot(t_arr, C_glic, 'b-', linewidth=2.5, label='$C_{glic}$ (líquida)')
ax1.plot(t_arr, q_glic, 'b--', linewidth=2, label='$q_{glic}$ (sólida)')
ax1.set_xlabel('Tempo [min]', fontsize=12)
ax1.set_ylabel('Concentração [mg/mL]', fontsize=12)
ax1.set_title('(a) Glicose: Fase Líquida vs. Sólida', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Fase líquida vs. sólida (frutose)
ax2.plot(t_arr, C_fru, 'r-', linewidth=2.5, label='$C_{fru}$ (líquida)')
ax2.plot(t_arr, q_fru, 'r--', linewidth=2, label='$q_{fru}$ (sólida)')
ax2.set_xlabel('Tempo [min]', fontsize=12)
ax2.set_ylabel('Concentração [mg/mL]', fontsize=12)
ax2.set_title('(b) Frutose: Fase Líquida vs. Sólida', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Distribuição entre Fases Líquida e Sólida',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

> 💡 **Observação:**
> - Inicialmente, toda a massa está na fase líquida (injeção em banda).
> - À medida que a adsorção ocorre, a massa migra para a fase sólida.
> - A frutose atinge maior concentração na fase sólida (maior afinidade).
> - Em equilíbrio, a razão $q/C = K$ é atingida para cada espécie.

In [ ]:
# ============================================================
# GRÁFICO 3: EFEITO DA SELETIVIDADE
# ============================================================

# Variar k1_fru para alterar a seletividade
alpha_values = [1.2, 1.5, 2.0, 3.0, 5.0]

fig, ax = plt.subplots(figsize=(12, 6))

cores = plt.cm.viridis(np.linspace(0, 1, len(alpha_values)))

for i, alpha_val in enumerate(alpha_values):
    k1_fru_val = alpha_val * K_glic * k2_fru   # Ajustar k1_fru para obter α desejado

    _, C_fru_val, _ = solver_euler_melhorado(
        y0_fru, k1_fru_val, k2_fru, t_final, dt)

    ax.plot(t_arr, C_fru_val, color=cores[i], linewidth=2,
            label=f'$\\alpha$ = {alpha_val:.1f}')

# Plotar glicose como referência
ax.plot(t_arr, C_glic, 'b--', linewidth=2, alpha=0.5, label='Glicose (referência)')

ax.set_xlabel('Tempo de retenção [min]', fontsize=12)
ax.set_ylabel('Concentração $C_{fru}$ [mg/mL]', fontsize=12)
ax.set_title('Efeito da Seletividade na Separação',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('📊 Efeito da seletividade:')
print('  • α ≈ 1: picos sobrepostos, separação impossível')
print('  • α ≈ 2: separação adequada (caso glicose/frutose)')
print('  • α > 3: separação fácil, picos bem afastados')

---
## 📈 Seção 8: Análise Quantitativa

In [ ]:
# ============================================================
# MÉTRICAS DE SEPARAÇÃO
# ============================================================

# Tempos de pico
t_pico_glic = t_arr[np.argmax(C_glic)]
t_pico_fru = t_arr[np.argmax(C_fru)]

# Alturas de pico
h_pico_glic = np.max(C_glic)
h_pico_fru = np.max(C_fru)

# Largura de pico (a meia altura)
meia_altura_glic = h_pico_glic / 2
meia_altura_fru = h_pico_fru / 2

# Resolução (medida de qualidade da separação)
# Rs = 2*(t2 - t1) / (w1 + w2), onde w é a largura na base
# Aproximação: w ≈ 4σ, onde σ é o desvio padrão do pico
sigma_glic = np.sqrt(np.sum((t_arr - t_pico_glic)**2 * C_glic) / np.sum(C_glic))
sigma_fru = np.sqrt(np.sum((t_arr - t_pico_fru)**2 * C_fru) / np.sum(C_fru))
w_glic = 4 * sigma_glic
w_fru = 4 * sigma_fru

Rs = 2 * (t_pico_fru - t_pico_glic) / (w_glic + w_fru)

print('📈 MÉTRICAS DE SEPARAÇÃO')
print('=' * 50)
print(f'\n📊 Glicose:')
print(f'  • Tempo de pico: {t_pico_glic:.3f} min')
print(f'  • Altura de pico: {h_pico_glic:.4f} mg/mL')
print(f'  • Largura (base): {w_glic:.3f} min')
print(f'\n📊 Frutose:')
print(f'  • Tempo de pico: {t_pico_fru:.3f} min')
print(f'  • Altura de pico: {h_pico_fru:.4f} mg/mL')
print(f'  • Largura (base): {w_fru:.3f} min')
print(f'\n📊 Qualidade da separação:')
print(f'  • Seletividade: α = {alpha_sel:.2f}')
print(f'  • Resolução: Rs = {Rs:.2f}')
print(f'  • Critério: Rs > 1,5 → separação completa')
if Rs > 1.5:
    print(f'  ✅ Separação COMPLETA (Rs = {Rs:.2f} > 1,5)')
else:
    print(f'  ⚠️ Separação PARCIAL (Rs = {Rs:.2f} ≤ 1,5)')

In [ ]:
# ============================================================
# TABELA COMPARATIVA
# ============================================================

dados_tabela = {
    'Grandeza': [
        'Constante de adsorção ($k_1$)',
        'Constante de dessorção ($k_2$)',
        'Afinidade de equilíbrio ($K$)',
        'Tempo de pico',
        'Altura de pico',
        'Largura de pico (base)',
        'Eluição'
    ],
    'Glicose': [
        f'{k1_glic} min⁻¹',
        f'{k2_glic} min⁻¹',
        f'{K_glic:.2f}',
        f'{t_pico_glic:.3f} min',
        f'{h_pico_glic:.4f} mg/mL',
        f'{w_glic:.3f} min',
        'Primeiro (menor afinidade)'
    ],
    'Frutose': [
        f'{k1_fru} min⁻¹',
        f'{k2_fru} min⁻¹',
        f'{K_fru:.2f}',
        f'{t_pico_fru:.3f} min',
        f'{h_pico_fru:.4f} mg/mL',
        f'{w_fru:.3f} min',
        'Depois (maior afinidade)'
    ]
}

df_comp = pd.DataFrame(dados_tabela)
display(Markdown('### Tabela Comparativa: Glicose vs. Frutose'))
display(Markdown(df_comp.to_markdown(index=False)))

---
## 🔍 Seção 9: Análise de Sensibilidade

Analisamos a sensibilidade da separação às constantes cinéticas $k_1$ e $k_2$.

In [ ]:
# ============================================================
# SENSIBILIDADE: CONSTANTES CINÉTICAS
# ============================================================

# Variar k2 (dessorção) mantendo K constante
k2_values = [0.5, 1.0, 2.0, 4.0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

cores = plt.cm.plasma(np.linspace(0, 1, len(k2_values)))

for i, k2_val in enumerate(k2_values):
    # Ajustar k1 para manter K constante
    k1_glic_val = K_glic * k2_val
    k1_fru_val = K_fru * k2_val

    _, C_glic_val, _ = solver_euler_melhorado(
        y0_glic, k1_glic_val, k2_val, t_final, dt)
    _, C_fru_val, _ = solver_euler_melhorado(
        y0_fru, k1_fru_val, k2_val, t_final, dt)

    ax1.plot(t_arr, C_glic_val, color=cores[i], linewidth=2,
             label=f'$k_2$ = {k2_val}')
    ax2.plot(t_arr, C_fru_val, color=cores[i], linewidth=2,
             label=f'$k_2$ = {k2_val}')

ax1.set_xlabel('Tempo [min]', fontsize=12)
ax1.set_ylabel('$C_{glic}$ [mg/mL]', fontsize=12)
ax1.set_title('(a) Glicose: Efeito de $k_2$ (K constante)',
              fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Tempo [min]', fontsize=12)
ax2.set_ylabel('$C_{fru}$ [mg/mL]', fontsize=12)
ax2.set_title('(b) Frutose: Efeito de $k_2$ (K constante)',
              fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle('Sensibilidade às Constantes Cinéticas (Afinidade K constante)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('📊 Sensibilidade às constantes cinéticas:')
print('  • Aumentar k2 (com K constante) acelera a dinâmica de adsorção/dessorção')
print('  • O tempo de pico NÃO muda (governado por K = k1/k2)')
print('  • A largura de pico DIMINUI (cinética mais rápida → menos espalhamento)')
print('  • Conclusão: K governa a posição do pico; k1 e k2 governam a largura')

> 💡 **Observação:**
> - **K governa a posição do pico** (tempo de retenção): espécies com maior K eluem mais tarde.
> - **$k_1$ e $k_2$ governam a largura do pico** (espalhamento): cinética mais rápida produz picos mais estreitos.
> - Essa separação de papéis é o que torna o cromatograma informativo: a partir dos dados experimentais, é possível estimar $K$ (pelo tempo de pico) e $k_1$, $k_2$ (pela largura de pico).
> - Este é o princípio do **problema inverso** discutido no Apêndice.

---
## 💡 Seção 10: Discussão

### Interpretação Física dos Resultados

1. **Mecanismo de separação:** A separação cromatográfica é possível porque a frutose tem maior afinidade pela fase estacionária ($K_{fru} = 1{,}09$) que a glicose ($K_{glic} = 0{,}54$). A seletividade $\alpha = K_{fru}/K_{glic} \approx 2{,}0$ garante que os picos fiquem suficientemente afastados para permitir coleta fracionada.

2. **Analogia física:** A coluna cromatográfica funciona como uma "corrida com obstáculos": as moléculas avançam na fase móvel, mas são periodicamente retidas na fase estacionária. Espécies com maior afinidade (frutose) param mais vezes e chegam ao final mais tarde.

3. **Cromatograma como impressão digital:** O tempo de pico codifica a afinidade $K_j$; a largura do pico codifica a cinética ($k_1$, $k_2$). Essa separação de papéis torna o cromatograma uma fonte rica de informação.

4. **Leito Móvel Simulado (LMS):** Em aplicações industriais, o processo contínuo é realizado em equipamentos LMS, que simulam o fluxo contracorrente por meio do chaveamento periódico das portas. O projeto do LMS requer a verificação da "região de separação" no plano das vazões (teoria do triângulo).

### Limitações do Modelo

- **Isoterma linear:** O modelo assume relação linear entre $C$ e $q$ (isoterma de Henry). Para concentrações altas, a isoterma pode ser não-linear (Langmuir, Freundlich).
- **Sem dispersão axial:** O modelo não inclui o espalhamento da banda por dispersão hidrodinâmica. Na prática, isso contribui para o alargamento dos picos.
- **Sem volumes mortos:** Não considera volumes extra-coluna (tubulações, detectores).
- **Sem gradientes de concentração na partícula:** Assume equilíbrio local na interface. Para partículas grandes, a difusão interna pode ser limitante.
- **Cinética de primeira ordem:** A transferência de massa real pode ser mais complexa (modelo de película, difusão intrapartícula).

### Aplicações Práticas

- **Indústria alimentícia:** Produção de xarope de milho de alta frutose (HFCS) por cromatografia preparativa.
- **Indústria farmacêutica:** Purificação de fármacos, anticorpos e biomoléculas.
- **Indústria química:** Separação de isômeros (xilenos, ácidos orgânicos).
- **Análise laboratorial:** Cromatografia analítica (HPLC) para identificação e quantificação de compostos.

### Conexão com a Teoria

> 📖 **Referência:** O modelo cinético de primeira ordem e a separação glicose/frutose são descritos em Oliveira et al. (2017) e Lugon Jr. et al. (2026), Cap. 12, Seção 12.4. O Leito Móvel Simulado segue Lugon Jr. et al. (2026), Cap. 12, Seção 12.5. A provocação sobre o problema inverso (estimação de $k_1$, $k_2$) é tratada no Apêndice de Problemas Inversos.

---
## 🔄 Seção 11: Navegação

- [📚 Voltar ao Capítulo 12](../notebooks/12_Adsorcao_Interfacial.ipynb)
- [💡 Caso 12.1 — Flotação por Espuma](EC_12_01_Adsorcao_Gas_Liquido_Flotacao_Espuma.ipynb)
- [📂 Outros Estudos de Caso](README.md)
- [🏠 Repositório Principal](../README.md)

---

<div align="center">
Estudo de Caso 12.2

Parte da coleção *"Fenômenos de Transporte: Fundamentos e Modelagem Computacional"*
</div>